In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# =========================================================================
# GMACs profiler for Infra-Bench CLS models -- CPU-only, no training data
# =========================================================================
# Loads each backbone from its canonical source, runs a single forward pass
# on a batch=16 dummy input, and reports GMACs and parameter count.
# Runtime: ~5-10 minutes on Colab CPU. No GPU needed. No dataset needed.
#
# Notes:
#   - fvcore counts multiply-accumulates as 1 FLOP each. We report GMACs
#     (Giga-MACs), which is 1 forward-pass MAC unit at batch=16.
#   - For models where fvcore can't trace (e.g. custom modules with weird
#     ops), we fall back to a rough parameter-based estimate marked with *.

!pip install -q fvcore olmoearth-pretrain-minimal 2>/dev/null

import torch
import torch.nn as nn
from fvcore.nn import FlopCountAnalysis
import numpy as np

BATCH = 16
IMAGE_SIZE = 224   # most FMs

results = {}

def report(name, model, dummy, note=''):
    """Runs one forward pass and reports GMACs + params."""
    model.eval()
    with torch.no_grad():
        try:
            flops = FlopCountAnalysis(model, dummy)
            # Suppress fvcore's per-module warnings by setting logging level
            flops.unsupported_ops_warnings(False).uncalled_modules_warnings(False)
            total_macs = flops.total()
            gmacs = total_macs / 1e9
            params = sum(p.numel() for p in model.parameters())
            results[name] = {'gmacs': gmacs, 'params_M': params/1e6, 'note': note}
            print(f'  {name:<24}  {gmacs:>8.2f} GMACs   {params/1e6:>7.1f}M params  {note}')
        except Exception as e:
            print(f'  {name:<24}  FAILED: {e}')
            results[name] = None
    del model, dummy


# ---- 1. SatlasPretrain S1 (Swin-B, 2-channel VH/VV, ~120x120 input) ----
print('\n[1/9] SatlasPretrain S1 (Sentinel1_SwinB_SI)')
try:
    import satlaspretrain_models as satlas
    model = satlas.Weights().get_pretrained_model('Sentinel1_SwinB_SI', fpn=False, head='classifier')
    dummy = torch.randn(BATCH, 2, 120, 120)  # VH, VV at native 120x120
    report('SatlasPretrain S1', model, dummy)
except Exception as e:
    print(f'  SatlasPretrain S1: SETUP FAILED - {e}')

# ---- 2. SatlasPretrain S2 (Swin-B, 9-channel with adapter, 224x224) ----
print('\n[2/9] SatlasPretrain S2 (Sentinel2_SwinB_SI_MS)')
try:
    model = satlas.Weights().get_pretrained_model('Sentinel2_SwinB_SI_MS', fpn=False, head='classifier')
    dummy = torch.randn(BATCH, 9, 224, 224)   # 9-channel input via channel-averaging adapter
    report('SatlasPretrain S2', model, dummy)
except Exception as e:
    print(f'  SatlasPretrain S2: SETUP FAILED - {e}')

# ---- 3. CROMA (joint S1+S2, ViT-B, 120x120) ----
print('\n[3/9] CROMA (CROMA_base)')
try:
    import os, urllib.request
    if not os.path.exists('use_croma.py'):
        urllib.request.urlretrieve(
            'https://raw.githubusercontent.com/antofuller/CROMA/main/use_croma.py',
            'use_croma.py')
    from use_croma import PretrainedCROMA
    from huggingface_hub import hf_hub_download
    ckpt = hf_hub_download(repo_id='antofuller/CROMA', filename='CROMA_base.pt')
    model = PretrainedCROMA(pretrained_path=ckpt, size='base', modality='both', image_resolution=120)
    # CROMA takes both s1 and s2 as separate tensors
    s1 = torch.randn(BATCH, 2, 120, 120)
    s2 = torch.randn(BATCH, 12, 120, 120)
    # fvcore requires positional tensor input for FlopCountAnalysis;
    # wrap so it takes a tuple
    class CROMAWrapper(nn.Module):
        def __init__(self, m): super().__init__(); self.m = m
        def forward(self, s1, s2): return self.m(SAR_images=s1, optical_images=s2)['joint_GAP']
    wrapper = CROMAWrapper(model)
    report('CROMA', wrapper, (s1, s2))
except Exception as e:
    print(f'  CROMA: SETUP FAILED - {e}')

# ---- 4. Prithvi-EO-2.0 (HF, ViT-L, 6 bands with T=1) ----
print('\n[4/9] Prithvi-EO-2.0-300M-TL')
try:
    from transformers import AutoModel
    model = AutoModel.from_pretrained('ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL',
                                       trust_remote_code=True, num_labels=0)
    dummy = torch.randn(BATCH, 6, 1, 224, 224)   # (B, C, T, H, W)
    class PrithviWrapper(nn.Module):
        def __init__(self, m): super().__init__(); self.m = m
        def forward(self, x): return self.m(x).last_hidden_state.mean(1)
    report('Prithvi-EO-2.0', PrithviWrapper(model), dummy)
except Exception as e:
    print(f'  Prithvi: SETUP FAILED - {e}')

# ---- 5. AlphaEarth Foundations (no backbone -- linear head only) ----
print('\n[5/9] AlphaEarth Foundations')
model = nn.Sequential(nn.Dropout(0.1), nn.Linear(64, 13))
dummy = torch.randn(BATCH, 64)
report('AlphaEarth', model, dummy, note='(linear head only; no backbone forward)')

# ---- 6. OlmoEarth v1.1-Base (ViT-B, 12-band S2 with T=1) ----
print('\n[6/9] OlmoEarth v1.1-Base')
try:
    from olmoearth_pretrain_minimal import load_model_from_id, ModelID, Modality
    model = load_model_from_id(ModelID.OLMOEARTH_V1_1_BASE)
    # Input as prepared by the notebook: (B, 12, T=1, H, W) reshape
    dummy = torch.randn(BATCH, 12, 1, 224, 224)
    # Simplified wrapper -- OlmoEarth's actual _prepare_input includes timestamps + mask;
    # this may under-count slightly if those add compute. Flag if the number looks off.
    class OlmoWrapper(nn.Module):
        def __init__(self, m): super().__init__(); self.m = m
        def forward(self, x):
            # Rearrange to (B, H, W, T, C) per OlmoEarth's convention if needed
            B, C, T, H, W = x.shape
            x_reshaped = x.permute(0, 3, 4, 2, 1)
            return self.m(x_reshaped)
    report('OlmoEarth v1.1-Base', OlmoWrapper(model), dummy,
           note='(may under-count; excludes timestamps/mask)')
except Exception as e:
    print(f'  OlmoEarth: SETUP FAILED - {e}')

# ---- 7. DINOv3 ViT-L/16 (HF, RGB, 224x224) ----
print('\n[7/9] DINOv3 ViT-L/16')
try:
    model = AutoModel.from_pretrained('facebook/dinov3-vitl16-pretrain-lvd1689m',
                                       trust_remote_code=True)
    dummy = torch.randn(BATCH, 3, 224, 224)  # RGB
    class DINOv3Wrapper(nn.Module):
        def __init__(self, m): super().__init__(); self.m = m
        def forward(self, x): return self.m(pixel_values=x).pooler_output
    report('DINOv3 ViT-L/16', DINOv3Wrapper(model), dummy)
except Exception as e:
    print(f'  DINOv3: SETUP FAILED - {e}')

# ---- 8. Supervised ResNet-18 (from-scratch, 9-channel adapter, 224x224) ----
print('\n[8/9] Supervised ResNet-18')
from torchvision.models import resnet18
model = resnet18(weights=None)
model.conv1 = nn.Conv2d(9, 64, kernel_size=7, stride=2, padding=3, bias=False)
model.fc = nn.Linear(512, 13)
dummy = torch.randn(BATCH, 9, 224, 224)
report('Supervised ResNet-18', model, dummy)

# ---- 9. Random Features ResNet-18 (identical arch to #8, frozen) ----
print('\n[9/9] Random Features (frozen ResNet-18)')
model = resnet18(weights=None)
model.conv1 = nn.Conv2d(9, 64, kernel_size=7, stride=2, padding=3, bias=False)
model.fc = nn.Linear(512, 13)
for p in model.parameters(): p.requires_grad = False
report('Random Features', model, dummy, note='(same arch as Sup RN-18)')

# ---- Summary ----
print('\n' + '=' * 80)
print(f'{"FM":<24}  {"GMACs":>10}  {"Params (M)":>12}')
print('=' * 80)
for name, r in results.items():
    if r is None: continue
    print(f'{name:<24}  {r["gmacs"]:>10.2f}  {r["params_M"]:>12.1f}')

In [ ]:
# =========================================================================
# Retry: GMACs for the 5 models that failed on the first pass.
# Uses timm architecture proxies where the official loader is fragile.
# Rationale: GMACs is a function of architecture, not trained weights.
# =========================================================================

!pip install -q timm fvcore 2>/dev/null

import torch, torch.nn as nn, timm
from fvcore.nn import FlopCountAnalysis

BATCH = 16

def report(name, model, dummy, note=''):
    model.eval()
    with torch.no_grad():
        try:
            flops = FlopCountAnalysis(model, dummy)
            flops.unsupported_ops_warnings(False).uncalled_modules_warnings(False)
            gmacs = flops.total() / 1e9
            params = sum(p.numel() for p in model.parameters())
            print(f'  {name:<28}  {gmacs:>8.2f} GMACs   {params/1e6:>7.1f}M params  {note}')
            return gmacs, params/1e6
        except Exception as e:
            print(f'  {name:<28}  FAILED: {e}')
            return None, None
    del model, dummy

# =========================================================================
# 1 + 2. SatlasPretrain S1 and S2 — Swin-B proxies
# Swin-B architecture (from timm) matches SatlasPretrain's backbone spec:
# 4-stage hierarchical transformer with the same channel counts.
# Only difference: adapter/first-conv, which is a tiny fraction of compute.
# =========================================================================
print('\n[1] SatlasPretrain S1 (Swin-B proxy, 2-channel input)')
# Use timm swin_base with 2 input channels; pretrained=False since we only need architecture
model = timm.create_model('swin_base_patch4_window7_224', pretrained=False, in_chans=2, num_classes=0)
report('SatlasPretrain S1', model, torch.randn(BATCH, 2, 224, 224),
       note='(Swin-B proxy; native input is 120x120 but Satlas uses 224x224 after resize)')
del model

print('\n[2] SatlasPretrain S2 (Swin-B proxy, 9-channel input)')
model = timm.create_model('swin_base_patch4_window7_224', pretrained=False, in_chans=9, num_classes=0)
report('SatlasPretrain S2', model, torch.randn(BATCH, 9, 224, 224), note='(Swin-B proxy)')
del model

# =========================================================================
# 4. Prithvi-EO-2.0-300M-TL — ViT-L/16 proxy with 6 input channels
# Standard ViT-L is 24 layers, 1024 dim, 16 heads. Prithvi adds a 3D patch
# embedding for temporal input but T=1 collapses this to standard 2D
# convolution; the extra compute is negligible (<0.1%).
# =========================================================================
print('\n[3] Prithvi-EO-2.0 (ViT-L/16 proxy, 6-channel input)')
model = timm.create_model('vit_large_patch16_224', pretrained=False, in_chans=6, num_classes=0)
report('Prithvi-EO-2.0', model, torch.randn(BATCH, 6, 224, 224),
       note='(ViT-L proxy; T=1 3D patch embed collapse is architecturally equivalent)')
del model

# =========================================================================
# 6. OlmoEarth v1.1-Base — ViT-B/16 proxy with 12 input channels
# Standard ViT-B is 12 layers, 768 dim, 12 heads.
# =========================================================================
print('\n[4] OlmoEarth v1.1-Base (ViT-B/16 proxy, 12-channel input)')
model = timm.create_model('vit_base_patch16_224', pretrained=False, in_chans=12, num_classes=0)
report('OlmoEarth v1.1-Base', model, torch.randn(BATCH, 12, 224, 224),
       note='(ViT-B/16 proxy; matches OlmoEarth architecture)')
del model

# =========================================================================
# 7. DINOv3 ViT-L/16 — timm ViT-L/16 proxy (bypasses HF gated repo)
# DINOv3 IS a standard ViT-L/16; the gated repo only restricts the
# pretrained WEIGHTS. GMACs are identical to timm's ViT-L/16 architecture.
# =========================================================================
print('\n[5] DINOv3 ViT-L/16 (timm ViT-L/16, 3-channel RGB input)')
model = timm.create_model('vit_large_patch16_224', pretrained=False, in_chans=3, num_classes=0)
report('DINOv3 ViT-L/16', model, torch.randn(BATCH, 3, 224, 224),
       note='(exact ViT-L/16 architecture; timm proxy bypasses HF gate)')
del model